In [12]:
# 1. ładowanie plików
import pandas as pd
import numpy as np
from collections import Counter
import re

# sciezka do pliku
file_path = "/home/batka/Desktop/Uganda/Project/project_selection_tables/Birds_primates_interactions.csv"

# wczytanie bez naglowkow
raw = pd.read_csv(file_path, header=None)

# kolumny z próbami
sample_cols = [
    c for c in range(5, raw.shape[1])
    if pd.notna(raw.iat[0, c])
]

print(raw.shape)
raw.head(10)

(59, 51)


,0,1,2,3,4,5,6,7,8,9,...,41,42,43,44,45,46,47,48,49,50
0,Feeding guild,Feeding guild2,Scientific name,Plot number,N,14_08A_CH,14_08A_CON1,14_08A_GM,14_08A_CON,14_08A_RC,...,20_08A_CON2,20_08A_BC,20_08A_CON3,20_08A_RCO,20_08B_RC,20_08B_CON1,20_08B_RCO,20_08B_CON2,20_08B_BC,20_08B_CON3
1,NaN,NaN,NaN,Day,NaN,14_08,14_08,14_08,14_08,14_08,...,20_08,20_08,20_08,20_08,20_08,20_08,20_08,20_08,20_08,20_08
2,NaN,NaN,NaN,Net,NaN,A,A,A,A,A,...,A,A,A,A,B,B,B,B,B,B
3,NaN,NaN,NaN,Stimuli,NaN,CH,CON1,GM,CON,RC,...,CON2,BC,CON3,RCO,RC,CON1,RCO,CON2,BC,CON3
4,NaN,NaN,NaN,Playback code,NaN,CH_1,SIL,OGM_2,SIL,ORC_2,...,SIL,NBC_1,SIL,RCO_4,NRC_5,SIL,RCO_3,SIL,NBC_3,SIL
5,NaN,NaN,NaN,Monkey present,NaN,GM,0,0,0,0,...,0,0,0,0,RTM,0,0,BC,BC,0
6,NaN,NaN,NaN,Start time [day_order],NaN,14_3,14_4,14_5,14_6,14_7,...,20_3,20_4,20_5,20_6,20_1,20_2,20_3,20_4,20_5,20_6
7,NaN,NaN,NaN,Start time [s from the start of the recording],NaN,08:45,09:20,09:55,10:30,11:05,...,6123.770,8073.480,10135.085,12197.210,1971.741,4120.154,6181.721,8356.986,10392.969,12538.712
8,NaN,NaN,NaN,Species richness (total),NaN,6,9,4,4,3,...,7,6,6,5,3,4,3,5,4,4
9,NaN,NaN,NaN,Species richness (I),NaN,3,5,2,2,2,...,2,6,5,4,1,3,3,4,2,3


In [15]:
# 2. przeliczenie czasu na normalny czas zegarowy

selection_times_path = (
    "/home/batka/Desktop/Uganda/Project/project_selection_tables/"
    "selection_times.csv"
)

selection_times = pd.read_csv(selection_times_path)

# znajdź odpowiednie wiersze w surowej tabeli
time_row = raw.index[
    raw[3].astype(str).str.strip()
    == "Start time [s from the start of the recording]"
][0]

day_row = raw.index[
    raw[3].astype(str).str.strip() == "Day"
][0]

net_row = raw.index[
    raw[3].astype(str).str.strip() == "Net"
][0]


# przypisanie recorderów do dnia i siatki
recording_id_map = {
    ("18_08", "A"): "1808_netA",
    ("18_08", "B"): "1808_netB",
    ("20_08", "A"): "20_08_bottom_net",
    ("20_08", "B"): "20_08_top_net"
}


# godzina rozpoczęcia każdego nagrania
recording_start_map = dict(
    zip(
        selection_times["Recording_id"],
        selection_times["Recording_start"]
    )
)


def to_clock_time(value, day, net):

    if pd.isna(value):
        return value

    value = str(value).strip()

    # jeśli czas jest już zapisany normalnie, np. 08:45
    if ":" in value:

        time = pd.to_datetime(
            value,
            format="%H:%M",
            errors="coerce"
        )

        return f"{time.hour}:{time.minute:02d}"

    # jeśli czas jest zapisany w sekundach od początku nagrania
    recording_id = recording_id_map[
        (str(day).strip(), str(net).strip())
    ]

    recording_start = pd.to_datetime(
        recording_start_map[recording_id],
        format="%H:%M"
    )

    clock_time = (
        recording_start
        + pd.to_timedelta(float(value), unit="s")
    )

    # Recording_start jest podany tylko z dokładnością do minuty,
    # więc zaokrąglamy wynik do najbliższej minuty
    clock_time = clock_time.round("min")

    return f"{clock_time.hour}:{clock_time.minute:02d}"


# przelicz czas we wszystkich kolumnach prób
for c in sample_cols:

    raw.iat[time_row, c] = to_clock_time(
        raw.iat[time_row, c],
        raw.iat[day_row, c],
        raw.iat[net_row, c]
    )


# sprawdzenie
print(
    raw.iloc[
        [day_row, net_row, time_row],
        sample_cols
    ]
)

# ujednolicenie czasu w obrębie par

# znajdź wiersz z pair_id
pair_row = raw.index[
    raw[3].astype(str).str.strip()
    == "Start time [day_order]"
][0]


def time_to_minutes(x):
    """Zmiana np. 8:45 na liczbę minut od północy."""

    x = str(x).strip()

    hour, minute = map(int, x.split(":"))

    return hour * 60 + minute


def minutes_to_time(x):
    """Zmiana liczby minut od północy z powrotem na np. 8:45."""

    hour = int(x // 60)
    minute = int(x % 60)

    return f"{hour}:{minute:02d}"


# wszystkie pair_id występujące w danych
pair_ids = raw.loc[
    pair_row,
    sample_cols
].dropna().unique()


for pair_id in pair_ids:

    # kolumny należące do tej samej pary
    pair_cols = [
        c for c in sample_cols
        if raw.iat[pair_row, c] == pair_id
    ]

    # czas obu prób w minutach od północy
    pair_times = [
        time_to_minutes(raw.iat[time_row, c])
        for c in pair_cols
    ]

    # średni czas pary
    mean_time = np.mean(pair_times)

    # zaokrąglenie do najbliższych 5 minut
    rounded_time = int(
        np.floor((mean_time + 2.5) / 5) * 5
    )

    # z powrotem do zapisu np. 9:05
    pair_time = minutes_to_time(rounded_time)

    # przypisz ten sam czas obu próbom w parze
    for c in pair_cols:
        raw.iat[time_row, c] = pair_time


# sprawdzenie
print(
    raw.iloc[
        [day_row, net_row, pair_row, time_row],
        sample_cols
    ]
)

      5      6      7      8      9      10     11     12     13     14  ...  \
1  14_08  14_08  14_08  14_08  14_08  14_08  14_08  14_08  14_08  14_08  ...   
2      A      A      A      A      A      B      B      B      B      B  ...   
7   8:45   9:20   9:55  10:30  11:05   8:45   9:20   9:55  10:30  11:05  ...   

      41     42     43     44     45     46     47     48     49     50  
1  20_08  20_08  20_08  20_08  20_08  20_08  20_08  20_08  20_08  20_08  
2      A      A      A      A      B      B      B      B      B      B  
7   8:49   9:22   9:56  10:30   7:35   8:11   8:45   9:21   9:55  10:31  

[3 rows x 46 columns]
      5      6      7      8      9      10     11     12     13     14  ...  \
1  14_08  14_08  14_08  14_08  14_08  14_08  14_08  14_08  14_08  14_08  ...   
2      A      A      A      A      A      B      B      B      B      B  ...   
6   14_3   14_4   14_5   14_6   14_7   14_3   14_4   14_5   14_6   14_7  ...   
7   8:45   9:20   9:55  10:30  11:05   8

In [17]:
# 3. funkcja do czyszczenia nazw

def clean_name(x):
    x = str(x).strip()
    x = x.replace(" ", "_")
    x = x.replace("-", "_")
    x = x.replace("/", "_")
    x = x.replace("%", "pct")
    x = x.replace("²", "2")
    x = x.replace("°", "deg")
    x = x.replace("μ", "u")
    x = re.sub(r"[()\[\]\.,:+']", "", x)
    x = re.sub(r"__+", "_", x)
    return x

# 4. wiersze metadanych i gatunków

meta_rows = list(range(0, 17))

# od wiersza 16 zaczynają się gatunki
species_rows = list(range(17, raw.shape[0]))

# 5. metadane 

meta_names = raw.iloc[meta_rows, 3].tolist()
meta_names[0] = "sample_id"

# czyszczenie nazw zmiennych
meta_names_clean = [
    clean_name(x).lower()
    for x in meta_names
]

# transpozycja:
# jedna próba = jeden wiersz
meta_df = raw.iloc[meta_rows, sample_cols].T.copy()
meta_df.columns = meta_names_clean

# krótsze nazwy najważniejszych zmiennych
meta_df = meta_df.rename(columns={
    "stimuli": "stimulus",
    "start_time_day_order": "pair_id",
    "start_time_s_from_the_start_of_the_recording": "time"
})

# 6. informacje o gatunkach

species_meta = raw.iloc[
    species_rows,
    [0, 1, 2, 3, 4]
].copy()

species_meta.columns = [
    "feeding_guild",
    "feeding_guild2",
    "scientific_name",
    "common_name",
    "N"
]

# usuń puste wiersze bez nazwy gatunku
species_meta = species_meta[
    species_meta["scientific_name"].notna()
].copy()

# odpowiadające indeksy w surowej tabeli
species_row_ids = species_meta.index.tolist()

# 7. nazwy kolumn gatunkowych

species_colnames = []
seen = {}

for _, row in species_meta.iterrows():

    species = row["scientific_name"]
    base = clean_name(species)

    # gdyby pojawiły się duplikaty nazw
    if base in seen:
        seen[base] += 1
        base = f"{base}_{seen[base]}"
    else:
        seen[base] = 1

    species_colnames.append(base)

# 8. tabela gatunki

species_df = raw.iloc[
    species_row_ids,
    sample_cols
].T.copy()

species_df.columns = species_colnames

# wartości gatunkowe powinny być 0/1
species_df = species_df.apply(
    pd.to_numeric,
    errors="coerce"
)

# 9. połączenie metadanych i gatunków

species_wide = pd.concat(
    [
        meta_df.reset_index(drop=True),
        species_df.reset_index(drop=True)
    ],
    axis=1
)

# wymuś unikalne nazwy kolumn w całej tabeli
cols = pd.Series(
    species_wide.columns,
    dtype="object"
)

for name in cols[cols.duplicated()].unique():

    idx = cols[cols == name].index.tolist()

    for j, k in enumerate(idx, start=1):
        cols.iloc[k] = f"{name}_{j}"

species_wide.columns = cols.tolist()

# 10. zmienne do późniejszej analizy

# kontrola vs okres po playbacku

species_wide["condition"] = np.where(
    species_wide["stimulus"]
    .astype(str)
    .str.startswith("CON"),
    "control",
    "post_playback"
)

# ustalenie, jaki playback odpowiada każdej parze
playback_lookup = (
    species_wide[
        species_wide["condition"] == "post_playback"
    ][["pair_id", "stimulus"]]
    .drop_duplicates()
    .rename(
        columns={"stimulus": "playback_species"}
    )
)

# przypisanie gatunku playbacku również do kontroli
species_wide = species_wide.merge(
    playback_lookup,
    on="pair_id",
    how="left"
)

# primate experiment vs positive control
species_wide["experiment"] = np.where(
    species_wide["playback_species"] == "RCO",
    "positive_control",
    "primate"
)

# 11. kolejność najważniejszych kolumn

first_cols = [
    "sample_id",
    "day",
    "pair_id",
    "net",
    "condition",
    "stimulus",
    "playback_species",
    "experiment",
    "playback_code",
    "monkey_present",
    "time",
    "recording_length_s",
    "performance_measure",
    "species_richness_total"
]

other_cols = [
    c for c in species_wide.columns
    if c not in first_cols
]

species_wide = species_wide[
    first_cols + other_cols
]

# 12. sprawdzenie danych

print("Wymiary tabeli:", species_wide.shape)

print(
    "Liczba próbek:",
    species_wide["sample_id"].nunique()
)

print(
    "Liczba gatunków:",
    len(species_colnames)
)

print("\nWarunki:")
print(
    species_wide["condition"].value_counts()
)

print("\nPlayback species:")
print(
    species_wide.loc[
        species_wide["condition"] == "post_playback",
        "playback_species"
    ].value_counts()
)

# 13. sprawdzenie par

pair_check = (
    species_wide
    .groupby(["pair_id", "condition"])
    .size()
    .unstack(fill_value=0)
)

print("\nSprawdzenie par:")
print(pair_check)

incorrect_pairs = pair_check[
    (pair_check["control"] != 1) |
    (pair_check["post_playback"] != 1)
]

print("\nNieprawidłowe pary:")
print(incorrect_pairs)

# 14. sprawdzenie species richness

# policz richness bezpośrednio z kolumn 0/1

species_wide["richness_calculated"] = (
    species_wide[species_colnames]
    .sum(axis=1)
)

species_wide["species_richness_total"] = pd.to_numeric(
    species_wide["species_richness_total"],
    errors="coerce"
)

# różnica między richness wpisanym w tabeli
# a richness policzonym z gatunków

species_wide["richness_difference_check"] = (
    species_wide["richness_calculated"] -
    species_wide["species_richness_total"]
)

print("\nSprawdzenie species richness:")
print(
    species_wide[
        [
            "sample_id",
            "species_richness_total",
            "richness_calculated",
            "richness_difference_check"
        ]
    ].head(10)
)

print(
    "\nLiczba próbek z niezgodnym richness:",
    (
        species_wide["richness_difference_check"] != 0
    ).sum()
)

# 15. podgląd

print(
    species_wide[
        [
            "sample_id",
            "day",
            "pair_id",
            "net",
            "condition",
            "stimulus",
            "playback_species",
            "species_richness_total"
        ]
    ].head(15)
)

# 16. zapis plików

out_csv = (
    "/home/batka/Desktop/Uganda/Project/project_selection_tables/"
    "Birds_primates_interactions_species_wide.csv"
)

species_wide.to_csv(
    out_csv,
    index=False
)

out_xlsx = (
    "/home/batka/Desktop/Uganda/Project/project_selection_tables/"
    "Birds_primates_interactions_species_wide.xlsx"
)

species_wide.to_excel(
    out_xlsx,
    index=False
)

# osobna tabela z informacjami o gatunkach
species_meta_out = (
    "/home/batka/Desktop/Uganda/Project/project_selection_tables/"
    "Birds_primates_interactions_species_metadata.csv"
)

species_meta.to_csv(
    species_meta_out,
    index=False
)

print("\nZapisano CSV:", out_csv)
print("Zapisano XLSX:", out_xlsx)
print("Zapisano species metadata:", species_meta_out)

Wymiary tabeli: (46, 62)
Liczba próbek: 46
Liczba gatunków: 42

Warunki:
condition
post_playback    23
control          23
Name: count, dtype: int64

Playback species:
playback_species
GM     7
RC     6
BC     4
CH     3
RCO    3
Name: count, dtype: int64

Sprawdzenie par:
condition  control  post_playback
pair_id                          
14_3             1              1
14_4             1              1
14_5             1              1
14_6             1              1
14_7             1              1
18_1             1              1
18_2             1              1
18_3             1              1
18_4             1              1
18_5             1              1
18_6             1              1
18_7             1              1
19_1             1              1
19_2             1              1
19_3             1              1
19_4             1              1
19_5             1              1
20_1             1              1
20_2             1              1
20_3        